In [ ]:
from google.colab import drive
drive.mount('/content/drive')

output_dirs = {
    "SNOMEDCT": "/content/drive/MyDrive/SNOMEDCT",
    "TMT": "/content/drive/MyDrive/TMT",
    "TMLT": "/content/drive/MyDrive/TMLT",
    "SCT": "/content/drive/MyDrive/SCT"
}

Mounted at /content/drive


SNOMEDCT+SCT NORMALIZE

In [ ]:
from google.colab import drive
import pandas as pd
import logging
import os
import re
import time

# ----------------------------- Logging -----------------------------------------
def setup_logging(verb: int = 1):
    level = logging.WARNING if verb <= 0 else (logging.INFO if verb == 1 else logging.DEBUG)
    logging.basicConfig(
        level=level,
        format="%(asctime)s %(levelname)s: %(message)s",
        datefmt="%H:%M:%S",
    )

# ----------------------------- Normalize Text ----------------------------------
def normalize_text(text: str) -> str:
    """Normalize the text (remove extra spaces and convert to lowercase, remove semanticTag)."""
    if not isinstance(text, str):
        return ""
    # Remove parentheses content for term_norm
    text = re.sub(r'\(.*?\)', '', text)  # Remove text in parentheses
    return text.strip().lower()

# ----------------------------- Extract Semantic Tag from Parentheses -------------------
def extract_semantic_tag_from_fsn(fsn: str) -> str:
    """Extract the semantic tag (last word in parentheses)."""
    if isinstance(fsn, str):
        match = re.search(r'\((.*?)\)', fsn)  # Search for text in parentheses
        if match:
            return match.group(1).strip()
    return "Unknown"

# ----------------------------- Extract Substance FSN (Main Substance) -------------------
def extract_substance_from_fsn(fsn: str) -> str:
    """Extract the main substance from FSN."""
    if isinstance(fsn, str):
        # Remove semantic tag and return the main substance term
        fsn = re.sub(r'\(.*?\)', '', fsn)  # Remove content in parentheses
        # Now return the main substance part (can adjust based on desired pattern)
        return fsn.strip()
    return "Unknown"

# ----------------------------- Merge & Process Data ----------------------------
def merge_snomed_sct(snomed_df, sct_df):
    print("Merging SCT with SNOMED-CT...")
    merged_df = pd.merge(sct_df, snomed_df, on="conceptId", how="left")
    print(f"Merged rows: {len(merged_df)}")

    # Filter out rows where isFSN is False, only keep rows with isFSN True
    merged_df = merged_df[merged_df["isFSN"] == True]
    print(f"Filtered rows (isFSN=True): {len(merged_df)}")

    # Drop unwanted columns: `term` and `isFSN`
    merged_df = merged_df.drop(columns=["term", "isFSN"])
    print("Dropped `term` and `isFSN` columns.")

    # Normalize term and create term_norm column
    merged_df["term_norm"] = merged_df["FSN"].apply(normalize_text)
    print("Normalized terms and created `term_norm`.")

    # Extract semanticTag from FSN (last word in parentheses)
    print("Extracting semanticTag from FSN...")
    merged_df["semanticTag"] = merged_df["FSN"].apply(extract_semantic_tag_from_fsn)
    print("Assigned semanticTag.")

    # Handle productFSN (should be the full FSN)
    merged_df["productFSN"] = merged_df["FSN"]
    print("Assigned productFSN.")

    # Extract main substance from FSN for substanceFSN
    print("Extracting substanceFSN from FSN...")
    merged_df["substanceFSN"] = merged_df["FSN"].apply(extract_substance_from_fsn)
    print("Assigned substanceFSN.")

    # Assign substanceConceptId based on id (if available)
    if 'id' in merged_df.columns:
        merged_df["substanceConceptId"] = merged_df["id"]
        print("Assigned substanceConceptId from id.")
    else:
        print("No 'id' column found, cannot assign substanceConceptId.")

    # Remove duplicates based on conceptId
    print("Removing duplicate rows based on conceptId...")
    merged_df = merged_df.drop_duplicates(subset=["conceptId"], keep="first")
    print(f"Rows after deduplication: {len(merged_df)}")

    # Drop `substanceFSN` column before saving to Excel
    merged_df = merged_df.drop(columns=["substanceFSN"])
    print("Dropped `substanceFSN` column before saving.")

    return merged_df

# ----------------------------- Save Results ------------------------------------
def save_to_excel(df, output_file):
    print(f"Saving results to {output_file} ...")
    df.to_excel(output_file, index=False)
    print("Save completed!")

# ----------------------------- Main Pipeline -----------------------------------
def run_pipeline(snomed_dir, sct_file, output_file):
    start_time = time.time()

    # Load SNOMED-CT files (from 0 to 6 parquet files in the directory)
    print("Loading SNOMED-CT files...")
    snomed_files = [os.path.join(snomed_dir, f) for f in os.listdir(snomed_dir) if f.endswith(".parquet")]
    snomed_df = pd.concat([pd.read_parquet(f) for f in snomed_files], ignore_index=True)
    print(f"Loaded {len(snomed_df)} SNOMED-CT rows from {len(snomed_files)} files.")

    # Load SCT file (single file)
    print("Loading SCT file...")
    sct_df = pd.read_parquet(sct_file)
    print(f"Loaded {len(sct_df)} SCT rows.")

    # Process merge
    merged_df = merge_snomed_sct(snomed_df, sct_df)

    # Save output
    save_to_excel(merged_df, output_file)
    print("Pipeline completed successfully!")

    # Measure time taken
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Time taken to execute: {execution_time} seconds")

# ----------------------------- Example Usage -----------------------------------
if __name__ == "__main__":
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Paths
    snomed_dir = "/content/drive/MyDrive/SNOMEDCT"
    sct_file = "/content/drive/MyDrive/SCT/SCT.parquet"
    output_file = "/content/drive/MyDrive/normalized_snomed_results.xlsx"

    # Run pipeline
    run_pipeline(snomed_dir, sct_file, output_file)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading SNOMED-CT files...
Loaded 623491 SNOMED-CT rows from 7 files.
Loading SCT file...
Loaded 361180 SCT rows.
Merging SCT with SNOMED-CT...
Merged rows: 615520
Filtered rows (isFSN=True): 359603
Dropped `term` and `isFSN` columns.
Normalized terms and created `term_norm`.
Extracting semanticTag from FSN...
Assigned semanticTag.
Assigned productFSN.
Extracting substanceFSN from FSN...
Assigned substanceFSN.
Assigned substanceConceptId from id.
Removing duplicate rows based on conceptId...
Rows after deduplication: 359602
Dropped `substanceFSN` column before saving.
Saving results to /content/drive/MyDrive/normalized_snomed_results.xlsx ...
Save completed!
Pipeline completed successfully!
Time taken to execute: 82.96805214881897 seconds


TMT NORMALIZE

In [ ]:
import pandas as pd
import re

# ฟังก์ชัน normalize สำหรับแปลงข้อความเป็นตัวพิมพ์เล็กและลบข้อมูลที่ไม่จำเป็น
def normalize_mastertmt(text: str, manufacturer: str) -> str:
    """Normalize text for MasterTMT (ลบคำที่มีคำเหมือนใน MANUFACTURER, ลบวงเล็บท้ายสุดและวงเล็บที่เหลือ)"""
    if not isinstance(text, str):
        return ""

    print(f"กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน MasterTMT สำหรับข้อความ: {text[:30]}...")  # แจ้งสถานะและแสดงข้อความส่วนต้น
    # ลบคำที่เหมือนใน MANUFACTURER (คำที่ตรงกับคำในคอลัมน์ MANUFACTURER)
    if manufacturer.lower() in text.lower():
        text = text.replace(manufacturer, "").strip()

    # ลบวงเล็บที่ท้ายสุด (เช่น TPU หรือสารสำคัญ)
    text = re.sub(r'\s?\([^\)]+\)\s*$', '', text)  # ลบวงเล็บที่อยู่ท้ายสุด
    text = text.strip()

    # ลบวงเล็บที่เหลือที่ไม่มีข้อมูล (เช่น ())
    text = re.sub(r'\s?\(\)\s*', '', text)  # ลบวงเล็บที่ไม่มีเนื้อหา
    text = text.strip()

    # แปลงข้อความทั้งหมดเป็นตัวพิมพ์เล็ก
    text = text.lower()

    return text

# ฟังก์ชัน normalize สำหรับ TP20250317 (ลบคำที่เหมือนใน MANUFACTURER, ลบวงเล็บท้ายสุดและวงเล็บที่เหลือ)
def normalize_tp20250317(text: str, manufacturer: str) -> str:
    """Normalize text for TP20250317 (ลบคำที่เหมือนใน MANUFACTURER, ลบวงเล็บท้ายสุดและวงเล็บที่เหลือ)"""
    if not isinstance(text, str):
        return ""

    print(f"กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: {text[:30]}...")  # แจ้งสถานะและแสดงข้อความส่วนต้น
    # ลบคำที่เหมือนใน MANUFACTURER (คำที่ตรงกับคำในคอลัมน์ MANUFACTURER)
    if manufacturer.lower() in text.lower():
        text = text.replace(manufacturer, "").strip()

    # ลบวงเล็บที่ท้ายสุด (เช่น TPU หรือสารสำคัญ)
    text = re.sub(r'\s?\([^\)]+\)\s*$', '', text)  # ลบวงเล็บที่อยู่ท้ายสุด
    text = text.strip()

    # ลบวงเล็บที่เหลือที่ไม่มีข้อมูล (เช่น ())
    text = re.sub(r'\s?\(\)\s*', '', text)  # ลบวงเล็บที่ไม่มีเนื้อหา
    text = text.strip()

    # แปลงข้อความทั้งหมดเป็นตัวพิมพ์เล็ก
    text = text.lower()

    return text

# โหลดไฟล์ MasterTMT
mastertmt_file = "/content/drive/MyDrive/TMT/MasterTMT_20250317.xlsx"
print(f"กำลังโหลดไฟล์ MasterTMT: {mastertmt_file}")
df_mastertmt = pd.read_excel(mastertmt_file)

# เลือกเฉพาะคอลัมน์ที่จำเป็น: FSN และ MANUFACTURER
df_mastertmt = df_mastertmt[['TPUCode', 'FSN', 'Manufacturer']]

# Normalize ข้อมูล FSN ใน MasterTMT
print("กำลัง Normalize ข้อมูล FSN ใน MasterTMT...")
df_mastertmt['FSN_norm'] = df_mastertmt.apply(lambda row: normalize_mastertmt(row['FSN'], row['Manufacturer']), axis=1)

# --- คลีนข้อมูล Concept ---

# กำหนดตัวแปร concept_texts และ concept_ids
concept_texts = []
concept_ids = []

# --- คลีนข้อมูล TP20250317 (ลบคำที่เหมือนใน MANUFACTURER, ลบวงเล็บท้ายสุดและวงเล็บที่เหลือ)
concept_files = [
    "/content/drive/MyDrive/TMT/Concept/TP20250317.xlsx"
]  # ตัวอย่างไฟล์ TP20250317

for concept_file in concept_files:
    print(f"กำลังโหลดไฟล์: {concept_file}")
    df_concept = pd.read_excel(concept_file)

    # เปลี่ยนชื่อคอลัมน์ TMTID ให้เป็น 'TMTID' จากชื่อคอลัมน์เดิม
    df_concept.rename(columns={df_concept.columns[0]: 'TMTID'}, inplace=True)  # แก้ไขให้คอลัมน์ TMTID มีชื่อเดียวกันทุกไฟล์

    # Normalize ข้อมูล FSN ใน TP20250317
    print("กำลัง Normalize ข้อมูล FSN ใน TP20250317...")
    df_concept['FSN_norm'] = df_concept.apply(lambda row: normalize_tp20250317(row['FSN'], row['MANUFACTURER']), axis=1)

    # ตรวจสอบความยาวของ concept_ids และ concept_texts ให้เท่ากันก่อนเพิ่มข้อมูล
    valid_indices = df_concept['FSN_norm'].dropna().index
    concept_texts.extend(df_concept.loc[valid_indices, 'FSN_norm'].tolist())
    concept_ids.extend(df_concept.loc[valid_indices, 'TMTID'].dropna().tolist())

# --- รวมข้อมูล MasterTMT และ Concept ---
print("กำลังรวมข้อมูล MasterTMT และ Concept...")
df_mastertmt['FSN_norm'] = df_mastertmt['FSN_norm'].dropna()
df_mastertmt['TMTID'] = df_mastertmt['TPUCode']  # เพิ่มข้อมูล id ใน MasterTMT

# ข้อมูลจาก Concept (รวม TMTID และ FSN_norm)
df_concept = pd.DataFrame({'TMTID': concept_ids, 'FSN_norm': concept_texts})

# รวมข้อมูล MasterTMT และ Concept
df_final = pd.concat([df_mastertmt[['TMTID', 'FSN_norm']], df_concept[['TMTID', 'FSN_norm']]], ignore_index=True)

# เซฟข้อมูลที่ normalize ลงไฟล์ Excel โดยไม่รวมคอลัมน์ FSN
output_file = "/content/drive/MyDrive/normalized_tmt_results.xlsx"
df_final.to_excel(output_file, index=False)

print(f"✅ ข้อมูลถูกเซฟที่: {output_file}")

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (ASTRAZENECA PHARMACEU...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (ASTRAZENECA PHARMACEU...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (ASTRAZENECA PHARMACEU...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (ASTRAZENECA PHARMACEU...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (BRISTOL-MYERS SQUIBB,...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (BRISTOL-MYERS SQUIBB,...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (BRISTOL-MYERS SQUIBB,...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SPRYCEL (BRISTOL-MYERS SQUIBB,...
กำลังลบคำที่เหมือนใน MANUFACTURER และวงเล็บใน TP20250317 สำหรับข้อความ: SQUEEZE (พอนด์ เคมีคอล)

TMLT

In [ ]:
import pandas as pd
import re

# ฟังก์ชัน normalize สำหรับแปลงข้อความเป็นตัวพิมพ์เล็กและลบวงเล็บที่ท้ายสุด และลบสัญลักษณ์ที่ไม่จำเป็น
def normalize_tmlt_fsn(text: str) -> str:
    """Normalize text for TMLT (ลบวงเล็บที่ไม่จำเป็น, ลบสัญลักษณ์ที่ไม่จำเป็น และแปลงเป็นตัวพิมพ์เล็ก)"""
    if not isinstance(text, str):
        return ""

    # ลบ * ที่หน้าชื่อการทดสอบ
    text = re.sub(r'^\*', '', text).strip()  # ลบ * ที่หน้าชื่อการทดสอบ

    # ลบ [+/-] ที่อาจจะมีในชื่อการทดสอบ
    text = re.sub(r'\[\+\/-\]', '', text).strip()  # ลบ [+/-]

    # ลบวงเล็บที่มีคำในวงเล็บ (เช่น [Type])
    text = re.sub(r'\[([^\]]+)\]', r'\1', text)  # ลบวงเล็บและคงคำภายในวงเล็บ
    text = text.strip()

    # ลบวงเล็บที่ท้ายสุด
    text = re.sub(r'\s?\([^\)]+\)\s*$', '', text)  # ลบวงเล็บที่อยู่ท้ายสุด
    text = text.strip()

    # ลบวงเล็บที่เหลือที่ไม่มีข้อมูล (เช่น ())
    text = re.sub(r'\s?\(\)\s*', '', text)  # ลบวงเล็บที่ไม่มีเนื้อหา
    text = text.strip()

    # แปลงข้อความทั้งหมดเป็นตัวพิมพ์เล็ก
    text = text.lower()

    return text

# โหลดข้อมูลจาก TMLT_ITEM
tmlt_item_file = "/content/drive/MyDrive/TMLT/Concept/TMLT_ITEM20250303.xlsx"
print(f"กำลังโหลดไฟล์ TMLT_ITEM: {tmlt_item_file}")
df_tmlt_item = pd.read_excel(tmlt_item_file)

# โหลดข้อมูลจาก TMLT_PANEL
tmlt_panel_file = "/content/drive/MyDrive/TMLT/Concept/TMLT_PANEL20250303.xlsx"
print(f"กำลังโหลดไฟล์ TMLT_PANEL: {tmlt_panel_file}")
df_tmlt_panel = pd.read_excel(tmlt_panel_file)

# ทำการ normalize ข้อมูล FSN ใน TMLT_ITEM
df_tmlt_item['FSN_norm'] = df_tmlt_item['FSN'].apply(normalize_tmlt_fsn)

# ทำการ normalize ข้อมูล FSN ใน TMLT_PANEL
df_tmlt_panel['FSN_norm'] = df_tmlt_panel['FSN'].apply(normalize_tmlt_fsn)

# ตรวจสอบชื่อคอลัมน์ในทั้งสองไฟล์
print(f"ชื่อคอลัมน์ใน TMLT_ITEM: {df_tmlt_item.columns.tolist()}")
print(f"ชื่อคอลัมน์ใน TMLT_PANEL: {df_tmlt_panel.columns.tolist()}")

# รวมข้อมูลจากทั้งสองไฟล์ โดยเลือกคอลัมน์ที่มีอยู่ในไฟล์ TMLT
df_tmlt_combined = pd.concat([df_tmlt_item[['TMLT', 'FSN', 'FSN_norm']],
                              df_tmlt_panel[['TMLT', 'FSN', 'FSN_norm']]], ignore_index=True)

# แสดงตัวอย่างข้อมูลที่รวม
print("\n📌 ตัวอย่างข้อมูลรวม (TMLT_ITEM และ TMLT_PANEL):\n")
print(df_tmlt_combined.head(5))

# เซฟข้อมูลที่ normalize ลงไฟล์ Excel
output_file_tmlt = "/content/drive/MyDrive/normalized_tmlt_results.xlsx"
df_tmlt_combined.to_excel(output_file_tmlt, index=False)

print(f"✅ ข้อมูลถูกเซฟที่: {output_file_tmlt}")

กำลังโหลดไฟล์ TMLT_ITEM: /content/drive/MyDrive/TMLT/Concept/TMLT_ITEM20250303.xlsx
กำลังโหลดไฟล์ TMLT_PANEL: /content/drive/MyDrive/TMLT/Concept/TMLT_PANEL20250303.xlsx
ชื่อคอลัมน์ใน TMLT_ITEM: ['TMLT', 'FSN', 'CHANGEDATE', 'FSN_norm']
ชื่อคอลัมน์ใน TMLT_PANEL: ['TMLT', 'FSN', 'CHANGEDATE', 'FSN_norm']

📌 ตัวอย่างข้อมูลรวม (TMLT_ITEM และ TMLT_PANEL):

     TMLT                                           FSN  \
0  380037             *B cell crossmatch [+/-] in Blood   
1  380039  *B cell crossmatch [+/-] in Blood from Donor   
2  380036           *HLA-DQB1 [Type] by High resolution   
3  380032           *HLA-DRB1 [Type] by High resolution   
4  380033           *HLA-DRB3 [Type] by High resolution   

                                 FSN_norm  
0             b cell crossmatch  in blood  
1  b cell crossmatch  in blood from donor  
2        hla-dqb1 type by high resolution  
3        hla-drb1 type by high resolution  
4        hla-drb3 type by high resolution  
✅ ข้อมูลถูกเซฟที่: /conten

MAPPING

SAPBERT

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 32.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import torch
import faiss
import numpy as np
import time
import os

# --- ตั้งค่า device (GPU หรือ CPU) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- ติดตั้ง Faiss ---
!pip install faiss-cpu -qq
import faiss
print("Faiss installed and imported.")

# --- โหลด SAP-BERT โมเดล ---
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")  # เปลี่ยนเป็น SAP-BERT model ของคุณได้
model = AutoModel.from_pretrained("bert-base-uncased").to(device)
print("SAP-BERT model loaded and moved to device.")

# --- ฟังก์ชันสร้างเวกเตอร์และ normalize สำหรับ Cosine similarity ---
def get_vector(text, tokenizer, model, device):
    if not isinstance(text, str):
        return np.zeros(model.config.hidden_size, dtype=np.float32)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    vec = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().astype(np.float32)
    # normalize vector สำหรับ cosine similarity
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    return vec

# --- โหลดไฟล์ Excel ---
tmt_file = "/content/drive/MyDrive/normalized_tmt_results.xlsx"
tmlt_file = "/content/drive/MyDrive/normalized_tmlt_results.xlsx"
snomed_file = "/content/drive/MyDrive/normalized_snomed_results.xlsx"

df_tmt = pd.read_excel(tmt_file)
df_tmlt = pd.read_excel(tmlt_file)
df_snomed = pd.read_excel(snomed_file)

# --- กรอง SNOMED-CT ตาม semantic tag ---
snomed_tmt = df_snomed[df_snomed['semanticTag'].isin([
    'substance', 'medicinal product', 'medicinal product form', 'clinical drug'
])].reset_index(drop=True)

snomed_tmlt = df_snomed[df_snomed['semanticTag'].isin([
    'procedure', 'regimen/therapy'
])].reset_index(drop=True)

print(f"SNOMED-CT สำหรับ TMT: {snomed_tmt.shape[0]} entries")
print(f"SNOMED-CT สำหรับ TMLT: {snomed_tmlt.shape[0]} entries")

# --- สร้างเวกเตอร์ SNOMED-CT พร้อม cache ---
def create_vectors(df_snomed_subset, cache_file):
    if os.path.exists(cache_file):
        print(f"Loading vectors from {cache_file} ...")
        return np.load(cache_file)
    else:
        vectors = np.array([get_vector(text, tokenizer, model, device) for text in df_snomed_subset['term_norm'].tolist()])
        np.save(cache_file, vectors)
        print(f"Vectors saved to {cache_file}")
        return vectors

snomed_vectors_tmt = create_vectors(snomed_tmt, "/content/drive/MyDrive/snomed_vectors_tmt_sapbert.npy")
snomed_vectors_tmlt = create_vectors(snomed_tmlt, "/content/drive/MyDrive/snomed_vectors_tmlt_sapbert.npy")

# --- สร้าง Faiss Index แยก สำหรับ Cosine similarity (normalize แล้ว) ---
dimension = snomed_vectors_tmt.shape[1]

index_tmt = faiss.IndexFlatIP(dimension)
index_tmt.add(snomed_vectors_tmt)
print(f"Faiss index TMT created with {index_tmt.ntotal} vectors.")

index_tmlt = faiss.IndexFlatIP(dimension)
index_tmlt.add(snomed_vectors_tmlt)
print(f"Faiss index TMLT created with {index_tmlt.ntotal} vectors.")

# --- Mapping TMT ---
mapped_tmt_results = []
start_time = time.time()
for idx, row in df_tmt.iterrows():
    tmt_vector = get_vector(row['FSN_norm'], tokenizer, model, device).reshape(1, -1)
    distances, indices = index_tmt.search(tmt_vector, 1)
    snomed_row = snomed_tmt.iloc[indices[0][0]]
    mapped_tmt_results.append({
        'TMTCode': row['TMTID'],
        'Name': row['FSN_norm'],
        'conceptId': snomed_row['conceptId'],
        'substanceConceptId': snomed_row.get('substanceConceptId', None),
        'Similarity_Score': distances[0][0]  # Cosine similarity
    })
    if (idx + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        print(f"TMT processed {idx + 1}/{df_tmt.shape[0]} rows, elapsed: {elapsed:.2f}s")

df_mapped_tmt = pd.DataFrame(mapped_tmt_results)

# --- Mapping TMLT ---
mapped_tmlt_results = []
for idx, row in df_tmlt.iterrows():
    tmlt_vector = get_vector(row['FSN_norm'], tokenizer, model, device).reshape(1, -1)
    distances, indices = index_tmlt.search(tmlt_vector, 1)
    snomed_row = snomed_tmlt.iloc[indices[0][0]]
    mapped_tmlt_results.append({
        'TMLT_Code': row['TMLT'],
        'Name': row['FSN_norm'],
        'conceptId': snomed_row['conceptId'],
        'FSN': snomed_row['FSN'],
        'Similarity_Score': distances[0][0]  # Cosine similarity
    })
    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"TMLT processed {idx + 1}/{df_tmlt.shape[0]} rows, total elapsed: {elapsed:.2f}s")

df_mapped_tmlt = pd.DataFrame(mapped_tmlt_results)

# --- แสดงตัวอย่างผลลัพธ์ ---
print("\n📌 ตัวอย่างผลการแมป TMT:")
display(df_mapped_tmt.head())
print("\n📌 ตัวอย่างผลการแมป TMLT:")
display(df_mapped_tmlt.head())

# --- บันทึกผลลัพธ์ ---
output_file_tmt = "/content/drive/MyDrive/tmt_snomed_mapping_results_sapbert_cosine.xlsx"
output_file_tmlt = "/content/drive/MyDrive/tmlt_snomed_mapping_results_sapbert_cosine.xlsx"

df_mapped_tmt.to_excel(output_file_tmt, index=False)
df_mapped_tmlt.to_excel(output_file_tmlt, index=False)

elapsed_total = time.time() - start_time
print(f"\n✅ ผลการแมป TMT ถูกเซฟที่: {output_file_tmt}")
print(f"✅ ผลการแมป TMLT ถูกเซฟที่: {output_file_tmlt}")
print(f"⏱ เวลาที่ใช้ทั้งหมด: {elapsed_total:.2f} วินาที")

BIOBERT

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss
import os
import time

# --- ตั้งค่า device (GPU หรือ CPU) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- โหลด BioBERT สำหรับการจับคู่คำศัพท์ทางการแพทย์ (TMT + TMLT) ---
model_name = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print(f"{model_name} loaded and moved to device.")

# --- ฟังก์ชันสร้างเวกเตอร์และ normalize ---
def get_vector(text, tokenizer, model, device):
    if not isinstance(text, str) or len(text.strip()) == 0:
        return np.zeros(model.config.hidden_size, dtype=np.float32)
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    vec = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy().astype(np.float32)
    norm = np.linalg.norm(vec)
    if norm > 0:
        vec = vec / norm
    if vec.shape[0] != model.config.hidden_size:
        vec = np.zeros(model.config.hidden_size, dtype=np.float32)
    return vec

# --- โหลดไฟล์ Excel ---
tmt_file = "/content/drive/MyDrive/normalized_tmt_results.xlsx"
tmlt_file = "/content/drive/MyDrive/normalized_tmlt_results.xlsx"
snomed_file = "/content/drive/MyDrive/normalized_snomed_results.xlsx"

df_tmt = pd.read_excel(tmt_file)
df_tmlt = pd.read_excel(tmlt_file)
df_snomed = pd.read_excel(snomed_file)

# --- กรอง SNOMED-CT ตาม semantic tag ---
snomed_tmt = df_snomed[df_snomed['semanticTag'].isin([
    'substance', 'medicinal product', 'medicinal product form', 'clinical drug'
])].reset_index(drop=True)

snomed_tmlt = df_snomed[df_snomed['semanticTag'].isin([
    'procedure', 'regimen/therapy'
])].reset_index(drop=True)

print(f"SNOMED-CT สำหรับ TMT: {snomed_tmt.shape[0]} entries")
print(f"SNOMED-CT สำหรับ TMLT: {snomed_tmlt.shape[0]} entries")

# --- สร้างเวกเตอร์ SNOMED-CT พร้อม cache ---
def create_vectors(df_snomed_subset, cache_file):
    if os.path.exists(cache_file):
        print(f"Loading vectors from {cache_file} ...")
        return np.load(cache_file)
    else:
        vectors = np.array([get_vector(str(text), tokenizer, model, device) for text in df_snomed_subset['term_norm'].tolist()])
        np.save(cache_file, vectors)
        print(f"Vectors saved to {cache_file}")
        return vectors

snomed_vectors_tmt = create_vectors(snomed_tmt, "/content/drive/MyDrive/snomed_vectors_tmt_biobert_mnli.npy")
snomed_vectors_tmlt = create_vectors(snomed_tmlt, "/content/drive/MyDrive/snomed_vectors_tmlt_biobert_mnli.npy")

# --- สร้าง Faiss Index (IP หลัง normalize = Cosine similarity) ---
dimension = snomed_vectors_tmt.shape[1]

index_tmt = faiss.IndexFlatIP(dimension)
index_tmt.add(snomed_vectors_tmt)
print(f"Faiss index TMT created with {index_tmt.ntotal} vectors.")

index_tmlt = faiss.IndexFlatIP(dimension)
index_tmlt.add(snomed_vectors_tmlt)
print(f"Faiss index TMLT created with {index_tmlt.ntotal} vectors.")

# --- Mapping TMT ---
mapped_tmt_results = []
start_time = time.time()
for idx, row in df_tmt.iterrows():
    fsn_text = str(row.get('FSN_norm', ''))
    tmt_vector = get_vector(fsn_text, tokenizer, model, device).reshape(1, -1)
    distances, indices = index_tmt.search(tmt_vector, 1)
    snomed_row = snomed_tmt.iloc[indices[0][0]]
    similarity_score = distances[0][0]  # Cosine similarity

    mapped_tmt_results.append({
        'TMTCode': row['TMTID'],
        'Name': fsn_text,
        'conceptId': snomed_row['conceptId'],
        'substanceConceptId': snomed_row.get('substanceConceptId', None),
        'Similarity_Score': similarity_score
    })

    if (idx + 1) % 1000 == 0:
        elapsed = time.time() - start_time
        print(f"TMT processed {idx + 1}/{df_tmt.shape[0]} rows, elapsed: {elapsed:.2f}s")

df_mapped_tmt = pd.DataFrame(mapped_tmt_results)

# --- Mapping TMLT ---
mapped_tmlt_results = []
for idx, row in df_tmlt.iterrows():
    fsn_text = str(row.get('FSN_norm', ''))
    tmlt_vector = get_vector(fsn_text, tokenizer, model, device).reshape(1, -1)
    distances, indices = index_tmlt.search(tmlt_vector, 1)
    snomed_row = snomed_tmlt.iloc[indices[0][0]]
    similarity_score = distances[0][0]

    mapped_tmlt_results.append({
        'TMLT_Code': row['TMLT'],
        'Name': fsn_text,
        'conceptId': snomed_row['conceptId'],
        'FSN': snomed_row['FSN'],
        'Similarity_Score': similarity_score
    })

    if (idx + 1) % 100 == 0:
        elapsed = time.time() - start_time
        print(f"TMLT processed {idx + 1}/{df_tmlt.shape[0]} rows, total elapsed: {elapsed:.2f}s")

df_mapped_tmlt = pd.DataFrame(mapped_tmlt_results)

# --- แสดงตัวอย่างผลลัพธ์ ---
print("\n📌 ตัวอย่างผลการแมป TMT:")
display(df_mapped_tmt.head())
print("\n📌 ตัวอย่างผลการแมป TMLT:")
display(df_mapped_tmlt.head())

# --- บันทึกผลลัพธ์ ---
output_file_tmt = "/content/drive/MyDrive/tmt_snomed_mapping_results_biobert_mnli.xlsx"
output_file_tmlt = "/content/drive/MyDrive/tmlt_snomed_mapping_results_biobert_mnli.xlsx"

df_mapped_tmt.to_excel(output_file_tmt, index=False)
df_mapped_tmlt.to_excel(output_file_tmlt, index=False)

elapsed_total = time.time() - start_time
print(f"\n✅ ผลการแมป TMT ถูกเซฟที่: {output_file_tmt}")
print(f"✅ ผลการแมป TMLT ถูกเซฟที่: {output_file_tmlt}")
print(f"⏱ เวลาที่ใช้ทั้งหมด: {elapsed_total:.2f} วินาที")